# Annotation validation — R1-1 (LLM accuracy) & R1-2 (inter-annotator agreement)

This notebook validates the GPT-4o-mini wordplay classifier against human annotation and reports
inter-annotator agreement, addressing reviewer comments **R1-1** and **R1-2**.

**Data.** A stratified sample of **300 paper titles** (150 the LLM labelled *Yes*, 150 *No*), each
independently annotated *Yes / No* for Beatles song-title wordplay by three humans (**RK, MT, GB**)
and by the **LLM**. The **`Majority`** column is the majority vote of the three humans — the LLM is
deliberately excluded, so validating the LLM against `Majority` is not circular.

**Imputation.** MT marked 4 titles `?` (uncertain). Following the conservative rule *"not confident
it is wordplay ⇒ No"*, these (and any missing cell) are imputed to **No**, so every statistic uses
the full *n = 300*.

**Inputs / outputs**
- in:  `../annotation_dataset_completed.xlsx` (sheet `Annotations`) — falls back to the shared CSV
       below if the workbook is not present (e.g. a clone of just this public repo).
- out: `data/shared/wordplay_annotation_sample.csv` (shareable, R1-7) and
       `output/annotation_validation_results.csv` (all metrics for the rebuttal).

In [ ]:
import itertools, os
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, cohen_kappa_score

SRC        = "../annotation_dataset_completed.xlsx"
OUT_CSV    = "data/shared/wordplay_annotation_sample.csv"
OUT_RESULTS = "output/annotation_validation_results.csv"
RATERS = ["LLM", "RK", "MT", "GB", "Majority"]
HUMANS = ["RK", "MT", "GB"]
RNG    = np.random.default_rng(42)   # reproducible bootstrap
N_BOOT = 10_000

## Helper functions

Cohen's κ and precision/recall/F1 use `scikit-learn`. Fleiss' κ and Krippendorff's α are implemented here (transparent, and they avoid an extra dependency).

In [ ]:
def to_bin(s):
    """Series of 'Yes'/'No' -> int 1/0 (1 = Yes = wordplay)."""
    return (s == "Yes").astype(int).to_numpy()

def pct_agree(a, b):
    return float((a == b).mean())

def class_metrics(truth, pred):
    """Binary arrays (1=Yes positive). Returns the validation metrics dict."""
    tp = int(((pred == 1) & (truth == 1)).sum())
    fp = int(((pred == 1) & (truth == 0)).sum())
    fn = int(((pred == 0) & (truth == 1)).sum())
    tn = int(((pred == 0) & (truth == 0)).sum())
    prec = tp / (tp + fp) if (tp + fp) else np.nan
    rec  = tp / (tp + fn) if (tp + fn) else np.nan
    spec = tn / (tn + fp) if (tn + fp) else np.nan
    npv  = tn / (tn + fn) if (tn + fn) else np.nan
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) else np.nan
    return dict(accuracy=(tp + tn) / len(truth), precision=prec, npv=npv, recall=rec,
                specificity=spec, f1=f1, balanced_acc=(rec + spec) / 2,
                TP=tp, FP=fp, FN=fn, TN=tn)

def _counts(sub):
    """sub: DataFrame of Yes/No (rows=items, cols=raters) -> N x 2 [Yes, No] counts."""
    yes = (sub == "Yes").sum(axis=1).to_numpy()
    return np.column_stack([yes, sub.shape[1] - yes])

def fleiss_kappa(sub):
    """Fleiss' kappa for n raters x N items of Yes/No labels."""
    mat = _counts(sub).astype(float)
    N, _ = mat.shape
    n = mat.sum(axis=1)[0]
    p = mat.sum(axis=0) / (N * n)
    P = ((mat ** 2).sum(axis=1) - n) / (n * (n - 1))
    return (P.mean() - (p ** 2).sum()) / (1 - (p ** 2).sum())

def krippendorff_alpha(sub, cats=("Yes", "No")):
    """Nominal Krippendorff's alpha (handles per-item missing via pairwise pairs)."""
    idx = {c: i for i, c in enumerate(cats)}
    o = np.zeros((len(cats), len(cats)))
    for _, row in sub.iterrows():
        vals = [v for v in row if v in idx]
        m = len(vals)
        if m < 2:
            continue
        for a, c in itertools.permutations(vals, 2):     # ordered pairs i != j
            o[idx[a], idx[c]] += 1.0 / (m - 1)
    n_c = o.sum(axis=1); n = n_c.sum()
    Do = o.sum() - np.trace(o)                            # sum_{c!=k} o_ck
    De = n * n - (n_c ** 2).sum()                         # sum_{c!=k} n_c n_k
    return 1 - (n - 1) * Do / De

def bootstrap_ci(stat_fn, m, n=N_BOOT, rng=RNG):
    """Percentile 95% CI; stat_fn takes a resampled row-index array."""
    out = np.empty(n)
    for i in range(n):
        ix = rng.integers(0, m, m)
        try:
            out[i] = stat_fn(ix)
        except Exception:
            out[i] = np.nan
    lo, hi = np.nanpercentile(out, [2.5, 97.5])
    return float(lo), float(hi)

def kappa_landis_koch(k):
    bins = [(-1, 0, "poor"), (0, .20, "slight"), (.20, .40, "fair"),
            (.40, .60, "moderate"), (.60, .80, "substantial"), (.80, 1.01, "almost perfect")]
    return next(lbl for lo, hi, lbl in bins if lo < k <= hi)

results = []   # collected for the summary table
def record(section, comparison, metric, est, lo=np.nan, hi=np.nan):
    results.append(dict(section=section, comparison=comparison, metric=metric,
                        estimate=est, ci95_low=lo, ci95_high=hi))

## 1. Load, clean and impute

Keep the 300 real data rows (the workbook has a 7-row manual scratchpad at the bottom that is dropped by requiring `#`), strip whitespace, and impute `?`/missing → `No`. We then assert that `Majority` equals a clean 3-human majority on all 300 rows.

In [ ]:
SHARE = ["annotation_id", "Song name", "Article title", *RATERS]
if os.path.exists(SRC):
    raw = pd.ExcelFile(SRC).parse("Annotations")
    df = raw[raw["#"].notna()].copy()                       # 300 real data rows
    df["annotation_id"] = df["#"].astype(int)
    for c in RATERS:
        df[c] = df[c].astype("string").str.strip().replace("?", "No").fillna("No")
    df = df.reset_index(drop=True)                          # keep Notes cols for error analysis
    os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)
    df[SHARE].to_csv(OUT_CSV, index=False, encoding="utf-8")  # shareable CSV (8 cols only)
    print(f"Loaded workbook and wrote {OUT_CSV}")
else:
    df = pd.read_csv(OUT_CSV)                               # external repro: Notes absent
    print(f"Workbook absent — loaded shared CSV {OUT_CSV}")

assert set(np.unique(df[RATERS].values)) <= {"Yes", "No"}, "non Yes/No labels remain"
maj3 = df[HUMANS].apply(lambda r: "Yes" if (r == "Yes").sum() >= 2 else "No", axis=1)
assert (maj3 == df["Majority"]).all(), "Majority != 3-human majority"

print(f"rows: {len(df)}   (Majority == 3-human majority on all {len(df)})")
pd.DataFrame({c: df[c].value_counts() for c in RATERS}).T[["Yes", "No"]]

## 2. R1-1 — LLM validation against the human majority

Ground truth = `Majority` (3-human vote); positive class = *Yes* (wordplay). 95% CIs are percentile bootstrap over the 300 annotated titles.

In [ ]:
truth, pred = to_bin(df["Majority"]), to_bin(df["LLM"])
m = len(df)

cm = confusion_matrix(truth, pred, labels=[1, 0])
cm_df = pd.DataFrame(cm, index=["truth: Yes", "truth: No"], columns=["pred: Yes", "pred: No"])
print("Confusion matrix (LLM vs majority):")
print(cm_df, "\n")

mt = class_metrics(truth, pred)
print(f"TP={mt['TP']}  FP={mt['FP']}  FN={mt['FN']}  TN={mt['TN']}\n")
for name in ["accuracy", "precision", "npv", "recall", "specificity", "f1", "balanced_acc"]:
    lo, hi = bootstrap_ci(lambda ix, n=name: class_metrics(truth[ix], pred[ix])[n], m)
    record("R1-1", "LLM vs Majority", name, mt[name], lo, hi)
    print(f"  {name:13} {mt[name]:.3f}   95% CI [{lo:.3f}, {hi:.3f}]")

k = cohen_kappa_score(truth, pred)
k_lo, k_hi = bootstrap_ci(lambda ix: cohen_kappa_score(truth[ix], pred[ix]), m)
record("R1-1", "LLM vs Majority", "cohen_kappa", k, k_lo, k_hi)
print(f"  {'cohen_kappa':13} {k:.3f}   95% CI [{k_lo:.3f}, {k_hi:.3f}]  ({kappa_landis_koch(k)})")

> **Interpreting the stratified design.** The 300 titles were sampled 150/150 by the LLM's *predicted* label, not drawn at random from the corpus (where wordplay is rare). Two metrics condition on the predicted label and are therefore each estimated *within* a sampling stratum, so they stay unbiased for the full candidate set: **precision / PPV** (of titles flagged as wordplay, the share that is genuine) and **NPV** (of titles rejected, the share correctly rejected). Metrics that condition on the *true* label (**recall, specificity**) or on the overall class mix (**accuracy, F1, balanced accuracy, Cohen's κ**) are evaluated on the artificially balanced sample and are *conditional on the 1:1 design* — read them as validation-set summaries, not corpus-wide rates. A prevalence-corrected recall/accuracy can be obtained if the corpus counts of LLM-positive vs. LLM-negative candidates are supplied.

In [ ]:
# LLM vs each individual rater (cross-checks the manual scratchpad in the workbook)
print("LLM vs each rater:")
for h in HUMANS + ["Majority"]:
    a = to_bin(df[h])
    pa, kk = pct_agree(pred, a), cohen_kappa_score(a, pred)
    record("R1-1", f"LLM vs {h}", "pct_agreement", pa)
    record("R1-1", f"LLM vs {h}", "cohen_kappa", kk)
    print(f"  LLM vs {h:9}  agreement={pa:.3f}   Cohen κ={kk:.3f}")

### Error analysis — why the classifier over-flags (low precision)

The false positives are dominated by **misattributed wordplay**: the title is a genuine pun, but on a *different Beatles song*, a *different artist*, or a *non-Beatles source*, rather than on the song supplied in the prompt. A small model such as `gpt-4o-mini` detects the presence of wordplay reliably but struggles to condition strictly on the specific song/artist in its context window. The annotators recorded a reason for a subset of the false positives; the cell below lists them (requires the source workbook — the shared CSV omits the free-text notes).

In [ ]:
fp = df[(df["LLM"] == "Yes") & (df["Majority"] == "No")].copy()
note_cols = [c for c in ["Notes", "Notes_RK", "Notes_MT", "Notes_GB"] if c in df.columns]
print(f"False positives (LLM=Yes, majority=No): {len(fp)}")
if note_cols:
    fp["notes"] = fp[note_cols].apply(
        lambda r: " | ".join(str(x) for x in r if pd.notna(x) and str(x).strip()), axis=1)
    noted = fp[fp["notes"].str.strip() != ""]
    print(f"... with an annotator note: {len(noted)} "
          f"(almost all 'wordplay on a different song / artist / source')\n")
    for _, r in noted.iterrows():
        print(f'  [{int(r["annotation_id"]):>3}] {str(r["Song name"])[:22]:22} | {r["notes"][:78]}')
else:
    print("(notes unavailable — running from the shared CSV)")

## 3. R1-2 — inter-annotator agreement (RK, MT, GB)

Pairwise percent agreement and Cohen's κ, plus Fleiss' κ and Krippendorff's α across all three human annotators. All use the full *n = 300*.

In [ ]:
print("Pairwise (human-human):")
for x, y in itertools.combinations(HUMANS, 2):
    a, b = to_bin(df[x]), to_bin(df[y])
    pa = pct_agree(a, b)
    kk = cohen_kappa_score(a, b)
    lo, hi = bootstrap_ci(lambda ix: cohen_kappa_score(a[ix], b[ix]), m)
    record("R1-2", f"{x}-{y}", "pct_agreement", pa)
    record("R1-2", f"{x}-{y}", "cohen_kappa", kk, lo, hi)
    print(f"  {x}-{y}:  agreement={pa:.3f}   Cohen κ={kk:.3f}  95% CI [{lo:.3f}, {hi:.3f}]  ({kappa_landis_koch(kk)})")

sub = df[HUMANS]
fk = fleiss_kappa(sub)
fk_lo, fk_hi = bootstrap_ci(lambda ix: fleiss_kappa(sub.iloc[ix]), m)
ka = krippendorff_alpha(sub)
ka_lo, ka_hi = bootstrap_ci(lambda ix: krippendorff_alpha(sub.iloc[ix]), m)
record("R1-2", "RK,MT,GB", "fleiss_kappa", fk, fk_lo, fk_hi)
record("R1-2", "RK,MT,GB", "krippendorff_alpha", ka, ka_lo, ka_hi)
print(f"\n  Fleiss κ (3 raters):       {fk:.3f}  95% CI [{fk_lo:.3f}, {fk_hi:.3f}]  ({kappa_landis_koch(fk)})")
print(f"  Krippendorff α (nominal):  {ka:.3f}  95% CI [{ka_lo:.3f}, {ka_hi:.3f}]")

## 4. Summary table

All metrics in one tidy frame, written to `output/annotation_validation_results.csv` for the rebuttal.

In [ ]:
summary = pd.DataFrame(results)[["section", "comparison", "metric",
                                  "estimate", "ci95_low", "ci95_high"]]
os.makedirs(os.path.dirname(OUT_RESULTS), exist_ok=True)
summary.to_csv(OUT_RESULTS, index=False)
print(f"wrote {OUT_RESULTS}  ({len(summary)} rows)")
summary.round(3)